# Step 2 — Two-Stage ML Pipeline

**Thesis:** Drift-Aware Selective Updating of Two-Stage Tabular ML Pipelines  
**Goal:** Build a sklearn-compatible two-stage pipeline where:
- **Stage 1** handles preprocessing (QuantileTransformer, TargetEncoder, StandardScaler).
- **Stage 2** is the predictive model (LogisticRegression or XGBClassifier).

Each stage must be independently refittable — the core mechanism for selective updating.

Reference implementation: `drift_framework/pipeline/two_stage.py`

## 2.1 Setup

In [ ]:
import sys, os

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, f1_score

SEED = 42
print("Setup OK")

In [ ]:
# Load data from the framework loader (built in Notebook 01)
from drift_framework.data.loader import load_dataset

bundle = load_dataset("adult")

## 2.2 Stage 1 — Preprocessor

Routes each column to the appropriate transformer:
- `QuantileTransformer` → skewed numeric features (`|skewness| > threshold`)
- `StandardScaler` → non-skewed numeric features
- `TargetEncoder` → high-cardinality categoricals (unique values > threshold)
- Integer label encoding → low-cardinality categoricals

In [ ]:
from typing import Optional
from category_encoders import TargetEncoder
from sklearn.preprocessing import QuantileTransformer, StandardScaler


class Stage1Preprocessor:
    """Fit-and-transform preprocessor for tabular features."""

    def __init__(
        self,
        num_features: list,
        cat_features: list,
        skew_threshold: float = 1.0,
        high_card_thresh: int = 10,
        random_state: int = SEED,
    ):
        self.num_features = num_features
        self.cat_features = cat_features
        self.skew_threshold = skew_threshold
        self.high_card_thresh = high_card_thresh
        self.random_state = random_state

        self._skewed_cols: list = []
        self._normal_cols: list = []
        self._high_card_cols: list = []
        self._low_card_cols: list = []

        self._qt: Optional[QuantileTransformer] = None
        self._ss: Optional[StandardScaler] = None
        self._te: Optional[TargetEncoder] = None
        self._label_maps: dict = {}
        self._fitted = False

    def fit(self, X: pd.DataFrame, y: pd.Series) -> "Stage1Preprocessor":
        """Fit all transformers on training data."""
        # Classify numeric columns by skewness
        self._skewed_cols = []
        self._normal_cols = []
        for col in self.num_features:
            if col in X.columns:
                sk = X[col].dropna().skew()
                (self._skewed_cols if abs(sk) > self.skew_threshold else self._normal_cols).append(col)

        # Classify categorical columns by cardinality
        self._high_card_cols = []
        self._low_card_cols = []
        for col in self.cat_features:
            if col in X.columns:
                (self._high_card_cols if X[col].nunique() > self.high_card_thresh else self._low_card_cols).append(col)

        if self._skewed_cols:
            self._qt = QuantileTransformer(output_distribution="normal", random_state=self.random_state)
            self._qt.fit(X[self._skewed_cols].fillna(0))

        if self._normal_cols:
            self._ss = StandardScaler()
            self._ss.fit(X[self._normal_cols].fillna(0))

        if self._high_card_cols:
            self._te = TargetEncoder(cols=self._high_card_cols, smoothing=1.0)
            self._te.fit(X[self._high_card_cols].astype(str), y)

        for col in self._low_card_cols:
            uniques = sorted(X[col].dropna().astype(str).unique())
            self._label_maps[col] = {v: i for i, v in enumerate(uniques)}

        self._fitted = True
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        """Transform features using fitted transformers."""
        if not self._fitted:
            raise RuntimeError("Call fit() before transform().")

        parts = []

        if self._skewed_cols:
            qt_arr = self._qt.transform(X[self._skewed_cols].fillna(0))
            parts.append(pd.DataFrame(qt_arr, columns=self._skewed_cols, index=X.index))

        if self._normal_cols:
            ss_arr = self._ss.transform(X[self._normal_cols].fillna(0))
            parts.append(pd.DataFrame(ss_arr, columns=self._normal_cols, index=X.index))

        if self._high_card_cols:
            te_df = self._te.transform(X[self._high_card_cols].astype(str))
            te_df.index = X.index
            parts.append(te_df)

        for col in self._low_card_cols:
            encoded = X[col].astype(str).map(self._label_maps[col]).fillna(-1).astype(int)
            parts.append(encoded.rename(col).to_frame())

        return pd.concat(parts, axis=1) if parts else pd.DataFrame(index=X.index)

    def fit_transform(self, X, y):
        return self.fit(X, y).transform(X)

In [ ]:
# Fit Stage 1 and inspect what got assigned where
s1 = Stage1Preprocessor(
    num_features=bundle.num_features,
    cat_features=bundle.cat_features,
)
s1.fit(bundle.X_ref, bundle.y_ref)

print(f"Skewed numeric cols   ({len(s1._skewed_cols)}): {s1._skewed_cols}")
print(f"Normal numeric cols   ({len(s1._normal_cols)}): {s1._normal_cols}")
print(f"High-card cat cols    ({len(s1._high_card_cols)}): {s1._high_card_cols}")
print(f"Low-card cat cols     ({len(s1._low_card_cols)}): {s1._low_card_cols}")

In [ ]:
# Transform and inspect output
X_ref_transformed = s1.transform(bundle.X_ref)
print(f"Transformed shape: {X_ref_transformed.shape}")
print(f"All numeric: {(X_ref_transformed.dtypes != object).all()}")
X_ref_transformed.head(3)

## 2.3 Stage 2 — Predictive model

In [ ]:
from typing import Literal
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier


def build_model(model_type: Literal["logreg", "xgboost"], random_state: int = SEED):
    """Instantiate the Stage 2 predictive model."""
    if model_type == "logreg":
        return LogisticRegression(max_iter=1000, random_state=random_state, n_jobs=-1)
    elif model_type == "xgboost":
        return XGBClassifier(
            n_estimators=300, learning_rate=0.05, max_depth=6,
            subsample=0.8, colsample_bytree=0.8,
            eval_metric="logloss", random_state=random_state, n_jobs=-1,
        )
    else:
        raise ValueError(f"Unknown model_type '{model_type}'. Use 'logreg' or 'xgboost'.")

## 2.4 TwoStagePipeline — full pipeline with independent refitting

In [ ]:
class TwoStagePipeline:
    """
    Two-stage pipeline:
      Stage 1 = preprocessing (Stage1Preprocessor)
      Stage 2 = predictive model (LogisticRegression or XGBClassifier)

    Either stage can be refit independently to simulate selective updating after
    covariate shift (refit Stage 1) or concept drift (refit Stage 2).
    """

    def __init__(
        self,
        model_type: Literal["logreg", "xgboost"] = "xgboost",
        num_features: list = None,
        cat_features: list = None,
        skew_threshold: float = 1.0,
        high_card_thresh: int = 10,
        random_state: int = SEED,
    ):
        self.model_type = model_type
        self.num_features = num_features or []
        self.cat_features = cat_features or []
        self.random_state = random_state

        self.stage1 = Stage1Preprocessor(
            num_features=self.num_features,
            cat_features=self.cat_features,
            skew_threshold=skew_threshold,
            high_card_thresh=high_card_thresh,
            random_state=random_state,
        )
        self.stage2 = build_model(model_type, random_state)
        self._stage1_fitted = False
        self._stage2_fitted = False

    def fit_stage1(self, X: pd.DataFrame, y: pd.Series) -> "TwoStagePipeline":
        """Fit (or re-fit) Stage 1 preprocessing only."""
        self.stage1.fit(X, y)
        self._stage1_fitted = True
        return self

    def fit_stage2(self, X: pd.DataFrame, y: pd.Series) -> "TwoStagePipeline":
        """Fit (or re-fit) Stage 2 model only, using already-fitted Stage 1."""
        if not self._stage1_fitted:
            raise RuntimeError("Fit Stage 1 first via fit_stage1().")
        X_transformed = self.stage1.transform(X)
        self.stage2.fit(X_transformed, y)
        self._stage2_fitted = True
        return self

    def fit(self, X: pd.DataFrame, y: pd.Series) -> "TwoStagePipeline":
        """Fit both stages sequentially."""
        return self.fit_stage1(X, y).fit_stage2(X, y)

    def predict_proba(self, X: pd.DataFrame) -> np.ndarray:
        """Return class probability estimates of shape (n_samples, 2)."""
        if not (self._stage1_fitted and self._stage2_fitted):
            raise RuntimeError("Pipeline must be fully fitted before predict_proba().")
        return self.stage2.predict_proba(self.stage1.transform(X))

    def predict(self, X: pd.DataFrame) -> np.ndarray:
        """Return binary class predictions."""
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)

## 2.5 Train Model A: LogisticRegression

In [ ]:
pipe_logreg = TwoStagePipeline(
    model_type="logreg",
    num_features=bundle.num_features,
    cat_features=bundle.cat_features,
)
pipe_logreg.fit(bundle.X_ref, bundle.y_ref)

proba_lr = pipe_logreg.predict_proba(bundle.X_pre)[:, 1]
auc_lr = roc_auc_score(bundle.y_pre, proba_lr)
f1_lr = f1_score(bundle.y_pre, (proba_lr >= 0.5).astype(int))

print(f"LogisticRegression — AUC: {auc_lr:.4f}  F1: {f1_lr:.4f}")

## 2.6 Train Model B: XGBClassifier

In [ ]:
pipe_xgb = TwoStagePipeline(
    model_type="xgboost",
    num_features=bundle.num_features,
    cat_features=bundle.cat_features,
)
pipe_xgb.fit(bundle.X_ref, bundle.y_ref)

proba_xgb = pipe_xgb.predict_proba(bundle.X_pre)[:, 1]
auc_xgb = roc_auc_score(bundle.y_pre, proba_xgb)
f1_xgb = f1_score(bundle.y_pre, (proba_xgb >= 0.5).astype(int))

print(f"XGBClassifier       — AUC: {auc_xgb:.4f}  F1: {f1_xgb:.4f}")

## 2.7 Demonstrate independent refitting

Simulating two selective-update strategies:
- **Covariate shift response** → refit Stage 1 only (adapt the preprocessor to the new distribution)
- **Concept drift response** → refit Stage 2 only (update the model's decision boundary)

In [ ]:
# Strategy A: refit Stage 1 only (using post-drift data as new reference)
pipe_refit_s1 = TwoStagePipeline(
    model_type="xgboost",
    num_features=bundle.num_features,
    cat_features=bundle.cat_features,
)
# Full fit first (baseline)
pipe_refit_s1.fit(bundle.X_ref, bundle.y_ref)
auc_before = roc_auc_score(bundle.y_pre, pipe_refit_s1.predict_proba(bundle.X_pre)[:, 1])

# Refit Stage 1 on post-drift data — Stage 2 weights unchanged
pipe_refit_s1.fit_stage1(bundle.X_post, bundle.y_post)
auc_after_s1 = roc_auc_score(bundle.y_post, pipe_refit_s1.predict_proba(bundle.X_post)[:, 1])

print(f"AUC before refitting Stage 1 : {auc_before:.4f}")
print(f"AUC after refitting Stage 1  : {auc_after_s1:.4f}  (Stage 2 untouched)")

In [ ]:
# Strategy B: refit Stage 2 only (model sees new labels, preprocessor stays fixed)
pipe_refit_s2 = TwoStagePipeline(
    model_type="xgboost",
    num_features=bundle.num_features,
    cat_features=bundle.cat_features,
)
pipe_refit_s2.fit(bundle.X_ref, bundle.y_ref)

# Refit Stage 2 on post-drift data — Stage 1 frozen
pipe_refit_s2.fit_stage2(bundle.X_post, bundle.y_post)
auc_after_s2 = roc_auc_score(bundle.y_post, pipe_refit_s2.predict_proba(bundle.X_post)[:, 1])

print(f"AUC after refitting Stage 2 only : {auc_after_s2:.4f}  (Stage 1 unchanged)")

## 2.8 Sanity checks

In [ ]:
# AUC must be > 0.5 on clean data for both models
assert auc_lr > 0.5, f"LogisticRegression AUC {auc_lr:.4f} is not above chance"
assert auc_xgb > 0.5, f"XGBClassifier AUC {auc_xgb:.4f} is not above chance"

# predict_proba output shape must be (n_samples, 2)
assert proba_xgb.ndim == 1  # we sliced [:, 1] already

# Refitting Stage 1 must not change Stage 2 model weights
from drift_framework.pipeline.two_stage import TwoStagePipeline as FWPipeline
fw_pipe = FWPipeline(
    model_type="xgboost",
    num_features=bundle.num_features,
    cat_features=bundle.cat_features,
)
fw_pipe.fit(bundle.X_ref, bundle.y_ref)
fw_proba = fw_pipe.predict_proba(bundle.X_pre)[:, 1]
fw_auc = roc_auc_score(bundle.y_pre, fw_proba)

# Both implementations should produce the same AUC
assert abs(auc_xgb - fw_auc) < 1e-4, \
    f"Notebook XGB AUC {auc_xgb:.4f} differs from framework AUC {fw_auc:.4f}"

print(f"LogReg  AUC {auc_lr:.4f} > 0.5 — OK")
print(f"XGBoost AUC {auc_xgb:.4f} > 0.5 — OK")
print(f"XGBoost AUC matches framework — OK")
print("\nAll sanity checks passed!")

## 2.9 Summary

| Model | AUC | F1 |
|---|---|---|
| LogisticRegression | `auc_lr` | `f1_lr` |
| XGBClassifier | `auc_xgb` | `f1_xgb` |

In [ ]:
summary = pd.DataFrame([
    {"model": "LogisticRegression", "auc": auc_lr, "f1": f1_lr},
    {"model": "XGBClassifier",      "auc": auc_xgb, "f1": f1_xgb},
])
summary